# Reporte Técnico: API de Análisis de Sentimiento (Modelo Híbrido G68)

**Equipo:** G68 - Hospitality Intelligence
**Fecha:** Enero 2026

## 1. Introducción y Propuesta de Valor
El objetivo es proveer al sector hotelero una herramienta que no solo clasifique el sentimiento, sino que identifique disparadores críticos de abandono de clientes. Priorizamos el **Recall Negativo** (Vetos) para asegurar que ninguna queja pase desapercibida.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
import sys
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, confusion_matrix
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Asegurar que encuentre la carpeta src para el motor híbrido
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))
from engine.sentiment_engine import analizar_sentimiento_hibrido

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Carga de Datos
data_path = "../data/raw/Big_AHR.csv" if os.path.exists("../data/raw/Big_AHR.csv") else "Big_AHR.csv"

if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    df.dropna(subset=['review_text'], inplace=True)
    df.drop_duplicates(subset=['review_text'], inplace=True)
    print(f"Datos cargados: {len(df)} registros.")
else:
    print("Utilizando datos de prueba.")
    df = pd.DataFrame({
        'rating': [5]*100 + [1]*20 + [3]*20,
        'review_text': ['Excelente estancia']*100 + ['Sucio y con moho']*20 + ['Normalito']*20
    })


Datos cargados: 13330 registros.


## 2. Entrenamiento y Calibración G68
Entrenamos el modelo base de Machine Learning.

In [2]:
df['sentiment'] = df['rating'].apply(lambda r: 'Negativo' if r<=2 else ('Neutro' if r==3 else 'Positivo'))

# Balanceo
df_maj = df[df.sentiment=='Positivo']
df_min_neg = resample(df[df.sentiment=='Negativo'], replace=True, n_samples=len(df_maj), random_state=42)
df_min_neu = resample(df[df.sentiment=='Neutro'], replace=True, n_samples=len(df_maj), random_state=42)
df_bal = pd.concat([df_maj, df_min_neg, df_min_neu])

X_train, X_test, y_train, y_test = train_test_split(df_bal['review_text'], df_bal['sentiment'], test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = CalibratedClassifierCV(LinearSVC(class_weight='balanced'), method='sigmoid')
model.fit(X_train_vec, y_train)
print("Modelo G68 listo.")

c:\ALURA - ONE\1. CIENCIA DE DATOS\HACKATHON\sentiment-api-G68\.venv\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\ALURA - ONE\1. CIENCIA DE DATOS\HACKATHON\sentiment-api-G68\.venv\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\ALURA - ONE\1. CIENCIA DE DATOS\HACKATHON\sentiment-api-G68\.venv\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\ALURA - ONE\1. CIENCIA DE DATOS\HACKATHON\sentiment-api-G68\.venv\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will 

Modelo G68 listo.


## 3. Rendimiento y Estabilidad (Stress Test)
Resultados del mega-stress test realizado sobre 2000 ejecuciones masivas.

- **Latencia Media:** 3.77 ms (Extremadamente rápido).
- **Latencia P95:** 5.28 ms.
- **Uso de RAM:** ~154 MB.
- **Precisión (Casos Críticos):** 70% (con 100% en niveles de sarcasmo directo).

## 4. Playground Interactivo G68
Ingresa una reseña para visualizar el análisis del motor híbrido.

In [4]:
text_input = widgets.Textarea(
    value='Excelente que el aire no funcionara y hubiera cucarachas.',
    placeholder='Escribe una reseña aquí...',
    description='Reseña:',
    layout=widgets.Layout(width='100%', height='100px')
)
button = widgets.Button(description="Analizar Sentimiento", button_style='primary')
output = widgets.Output()

def on_button_clicked(b):
    with output:
        clear_output()
        test_review = text_input.value
        if not test_review.strip():
            print("Por favor ingresa un texto.")
            return
            
        prevision, prob, meta = analizar_sentimiento_hibrido(test_review, model, vectorizer)
        
        # Estilo de color según sentimiento
        color = "#2ecc71" if prevision == "Positivo" else ("#e74c3c" if prevision == "Negativo" else "#f1c40f")
        
        display(HTML(f"""
        <div style="border: 2px solid {color}; padding: 15px; border-radius: 10px; background-color: #f9f9f9;">
            <h3 style="color: {color}; margin-top: 0;">Resultado: {prevision} ({prob:.2%})</h3>
            <p><b>Tipo de Voto:</b> {meta.get('votos', 'N/A')}</p>
            <p><b>Score Reglas:</b> {meta.get('score_reglas', 0.0):.3f}</p>
            <p><b>Triggers Detectados:</b> <span style="color: #e67e22;">{', '.join(meta.get('explicabilidad', {}).get('triggers', []))}</span></p>
            <p><b>Áreas Afectadas:</b> <span style="color: #9b59b6;">{', '.join(meta.get('explicabilidad', {}).get('areas', []))}</span></p>
        </div>
        """))

button.on_click(on_button_clicked)
display(text_input, button, output)

Textarea(value='Excelente que el aire no funcionara y hubiera cucarachas.', description='Reseña:', layout=Layo…

Button(button_style='primary', description='Analizar Sentimiento', style=ButtonStyle())

Output()

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
import sys
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, confusion_matrix
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Asegurar que encuentre la carpeta src para el motor híbrido
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))
from engine.sentiment_engine import analizar_sentimiento_hibrido

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Carga de Datos
data_path = "../data/raw/Big_AHR.csv" if os.path.exists("../data/raw/Big_AHR.csv") else "Big_AHR.csv"

if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    df.dropna(subset=['review_text'], inplace=True)
    df.drop_duplicates(subset=['review_text'], inplace=True)
    print(f"Datos cargados: {len(df)} registros.")
else:
    print("Utilizando datos de prueba.")
    df = pd.DataFrame({
        'rating': [5]*100 + [1]*20 + [3]*20,
        'review_text': ['Excelente estancia']*100 + ['Sucio y con moho']*20 + ['Normalito']*20
    })


Datos cargados: 13330 registros.


In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
import sys
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, confusion_matrix
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Asegurar que encuentre la carpeta src para el motor híbrido
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))
from engine.sentiment_engine import analizar_sentimiento_hibrido

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Carga de Datos
data_path = "../data/raw/Big_AHR.csv" if os.path.exists("../data/raw/Big_AHR.csv") else "Big_AHR.csv"

if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    df.dropna(subset=['review_text'], inplace=True)
    df.drop_duplicates(subset=['review_text'], inplace=True)
    print(f"Datos cargados: {len(df)} registros.")
else:
    print("Utilizando datos de prueba.")
    df = pd.DataFrame({
        'rating': [5]*100 + [1]*20 + [3]*20,
        'review_text': ['Excelente estancia']*100 + ['Sucio y con moho']*20 + ['Normalito']*20
    })


Datos cargados: 13330 registros.
